## **Time Series Analysis Task Notebook**

This notebook is designed to test you through various Time Series Analysis tasks using the Bike Sharing dataset. The tasks will help you develop essential skills in handling time-based data, including cleaning and preprocessing, feature engineering, and model building. You'll explore techniques such as creating lag and rolling window features, implementing linear regression for time series prediction, and applying XGBoost with hyperparameter tuning. Finally, you'll evaluate and compare the performance of the models, providing insights into their effectiveness. These tasks will enhance your understanding of time series analysis and prepare you for real-world forecasting challenges.

# **About the Dataset**
The Bike Sharing Dataset contains information about bike rental counts in the city of Washington, D.C., recorded at hourly intervals. The dataset provides data on various factors that influence bike rentals, including weather conditions, time of day, and seasonal factors. It includes several features such as temperature, humidity, wind speed, and day of the week, which can be used for building predictive models to forecast bike rental demand.

The dataset spans multiple years and captures the number of bikes rented at each hour of the day, making it ideal for time series analysis. By leveraging this data, you can explore patterns in bike rentals over time, uncover seasonality, and implement models for predicting future rental demand.





## **Key Attributes in the Dataset:**

**instant:** Record index.

**dteday:** Date of the observation.

**season:** The season (1: Spring, 2: Summer, 3: Fall, 4: Winter).

**yr:** Year (0: 2011, 1: 2012).

**mnth:** Month of the year (1 to 12).

**hour:** Hour of the day (0 to 23).

**holiday:** Whether the day is a holiday (1: Yes, 0: No).

**weekday:** Day of the week (0 to 6).

**workingday:** Whether it's a working day (1: Yes, 0: No).

**weathersit:** Weather condition (1: Clear, 2: Mist, 3: Light Rain, 4: Heavy Rain).

**temp:** Temperature (normalized).

**hum:** Humidity (normalized).

**windspeed:** Wind speed (normalized).

**cnt:** The total number of bike rentals at that hour.

This dataset offers a comprehensive set of features to explore and analyze bike-sharing patterns, making it an excellent resource for time series forecasting tasks.

## **Exercise**

1. Load the [dataset](https://www.kaggle.com/datasets/lakshmi25npathi/bike-sharing-dataset) from Kaggle. Use the "hour.csv" file.

In [ ]:
import os
import kagglehub

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
color_pal = sns.color_palette()
plt.style.use('fivethirtyeight')

from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.linear_model import LinearRegression
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
path = kagglehub.dataset_download("lakshmi25npathi/bike-sharing-dataset")

print("Path to dataset files:", path)

1. Load the [dataset](https://www.kaggle.com/datasets/lakshmi25npathi/bike-sharing-dataset) from Kaggle. Use the "hour.csv" file.

In [ ]:
dataset_path = "C:/Users/nikol/.cache/kagglehub/datasets/lakshmi25npathi/bike-sharing-dataset/versions/1"

files = os.listdir(dataset_path)
files


In [ ]:
days_path = "C:/Users/nikol/.cache/kagglehub/datasets/lakshmi25npathi/bike-sharing-dataset/versions/1/day.csv"
hours_path = "C:/Users/nikol/.cache/kagglehub/datasets/lakshmi25npathi/bike-sharing-dataset/versions/1/hour.csv"

days_data = pd.read_csv(days_path)
hours_data = pd.read_csv(hours_path)

In [ ]:
print('Days dataset:' + '\n' + str(days_data.head()))
print("--------------------------------------------------------------------------------------")
print('Hours dataset:' + '\n' + str(hours_data.head()))

2. Visualize the structure of the dataset using appropriate libraries and plots.

In [ ]:
# Days dataset
sns.scatterplot(x='dteday', y='cnt', data=days_data)
plt.title('Bike Rentals per Day')
plt.show()

# Perform seasonal decomposition
result_days = seasonal_decompose(days_data['cnt'], model='additive', period=365)
result_days.plot()

# Hours dataset
sns.scatterplot(x='dteday', y='cnt', data=hours_data)
plt.title('Bike Rentals per Hour')
plt.show()

# Perform seasonal decomposition
result_hours = seasonal_decompose(hours_data['cnt'], model='additive', period=24)
result_hours.plot()

3. Clean and pre-process the dataset as required and prepare the data for modelling.


In [ ]:
# 3. Clean and pre-process the dataset as required and prepare the data for modelling.
hours_data['dteday'] = pd.to_datetime(hours_data['dteday'])
hours_data['hour'] = hours_data['hr']
hours_data['day_of_week'] = hours_data['dteday'].dt.dayofweek
hours_data['month'] = hours_data['dteday'].dt.month
hours_data['year'] = hours_data['dteday'].dt.year
hours_data.head()


4. Create the lag and rolling windows features for the "cnt" column such as: 1 day lag, 1 week lag, 1 month, etc. and last 3 day rolling mean, last 3 hours rolling mean, etc. But it should be based on your dataset and what makes sense for this dataset.


In [ ]:
#use shift(1), (7) (30), (365) to create lag features for the target variable 'cnt'
hours_data['cnt_lag_1'] = hours_data['cnt'].shift(1)
hours_data['cnt_lag_7'] = hours_data['cnt'].shift(7)
hours_data['cnt_lag_30'] = hours_data['cnt'].shift(30)
hours_data['cnt_lag_365'] = hours_data['cnt'].shift(365)

# rolling
hours_data['cnt_rolling_3'] = hours_data['cnt'].rolling(window=3).mean()
hours_data['cnt_rolling_7'] = hours_data['cnt'].rolling(window=7).mean()
hours_data['cnt_rolling_30'] = hours_data['cnt'].rolling(window=30).mean()
hours_data['cnt_rolling_365'] = hours_data['cnt'].rolling(window=365).mean()

hours_data

5. Implement linear regression to predict how many bikes will be rented each hour of the last week and evaluate using appropriate metrics.

In [27]:
features = ['hour', 'day_of_week', 'month', 'year', 'cnt_lag_1', 'cnt_lag_7', 'cnt_lag_30', 'cnt_lag_365',
            'cnt_rolling_3', 'cnt_rolling_7', 'cnt_rolling_30', 'cnt_rolling_365']
target = 'cnt'
train_data = hours_data.dropna()
X_train = train_data[features]
y_train = train_data[target]
model_lr = LinearRegression()
model_lr.fit(X_train, y_train)
# Predict bikes will be rented each hour of the last week
X_test = hours_data[features].tail(168)
y_test = hours_data[target].tail(168)
y_pred_lr = model_lr.predict(X_test)
# show
results = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_lr})
print(results)
# Evaluate the model
mse_lr = mean_squared_error(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
print(f"Linear Regression - MSE: {mse_lr}, MAE: {mae_lr}")

       Actual   Predicted
17211      11   48.970046
17212      13   23.037201
17213      13   19.092725
17214       7   16.085306
17215       1   14.454764
...       ...         ...
17374     119  115.314069
17375      89   73.325748
17376      90   85.184058
17377      61   71.248972
17378      49   83.781426

[168 rows x 2 columns]
Linear Regression - MSE: 640.9672149488459, MAE: 19.48648686507245


6. Implement XGBoost to predict how many bikes will be rented each hour of the last week and evaluate using appropriate metrics.

In [28]:
# 6. Implement XGBoost to predict how many bikes will be rented each hour of the last week and evaluate using appropriate metrics.
model_xgb = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1)
model_xgb.fit(X_train, y_train)
y_pred_xgb = model_xgb.predict(X_test) 
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
# show
results_xgb = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_xgb})
print(results_xgb)
print(f"XGBoost - MSE: {mse_xgb}, MAE: {mae_xgb}")

       Actual  Predicted
17211      11  13.547039
17212      13   6.321427
17213      13   7.559597
17214       7   6.029000
17215       1   4.762809
...       ...        ...
17374     119  98.771637
17375      89  76.122269
17376      90  72.917290
17377      61  58.827793
17378      49  49.325214

[168 rows x 2 columns]
XGBoost - MSE: 256.42449951171875, MAE: 9.766749382019043


7. Experiment with predicting different time periods, such as use all data to predict bike rentals for just the next day (24 hours) or the next entire month and then see how much better or worse the model gets

In [30]:
# 7. Experiment with predicting different time periods, such as use all data to predict bike rentals for just the next day (24 hours) or the next entire month and then see how much better or worse the model gets
# Predict bike rentals for the next day (24 hours)



Next Day Prediction - MSE: 951.1360473632812, MAE:19.253149032592773


8. Experiment tuning hyperparameters

**Bonus task (Optional)**

This tasks is not mandatory, but it is designed for those who want to challenge themselves, enhance their critical thinking skills, or dive deeper into the topic. If you're eager to learn more or test your understanding, this task can provide additional learning opportunities.
1. Modelling: Implement an ARIMA model, evaluate it using relevant plots and provide a summary analysis .
  
